In [8]:
# =========================
# Load data and show head
# =========================

import os
import random
import pandas as pd

TRAIN_CSV = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OUTPUT_DIR = "/kaggle/working/adapter_output"

RANDOM_SEED = 42
TRAIN_ROWS_LIMIT = 2000
VAL_FRACTION = 0.05

random.seed(RANDOM_SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(TRAIN_CSV)
print(df.head())

if TRAIN_ROWS_LIMIT is not None:
    df = df.iloc[:TRAIN_ROWS_LIMIT].copy()

df = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
val_size = max(1, int(len(df) * VAL_FRACTION))
val_df = df.iloc[:val_size].copy()
train_df = df.iloc[val_size:].copy()

print(f"train={len(train_df)} val={len(val_df)}")




         id                                             prompt  \
0  00066667  In Alice's Wonderland, a secret bit manipulati...   
1  000b53cf  In Alice's Wonderland, a secret bit manipulati...   
2  00189f6a  In Alice's Wonderland, secret encryption rules...   
3  001b24c4  In Alice's Wonderland, numbers are secretly co...   
4  001c63cb  In Alice's Wonderland, secret encryption rules...   

                  answer  
0               10010111  
1               01000011  
2      cat imagines book  
3                XXXVIII  
4  wizard creates secret  
train=1900 val=100


In [10]:
# =========================
# Create adapter
# =========================

import os
import site
import importlib.util
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)

# Re-add NVIDIA utility paths so mamba_ssm can see CUTLASS
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script")
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/")

MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0))
print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
print("cutlass spec:", importlib.util.find_spec("cutlass"))
print("mamba_ssm spec:", importlib.util.find_spec("mamba_ssm"))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    use_fast=False,
    local_files_only=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ---- config ----
SYSTEM_PROMPT = (
    "You are a careful mathematical reasoning assistant. "
    "For each test case, generate a response and place the final answer "
    "inside a single LaTeX \\boxed{} command."
)

MAX_PROMPT_TOKENS = 256
MAX_ANSWER_TOKENS = 64
MAX_TOTAL_TOKENS = 384

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
LOGGING_STEPS = 10
SAVE_STEPS = 100

PROMPT_TEMPLATE = (
    "Solve the following problem. Put the final answer inside \\boxed{}.\n\n"
    "Problem:\n{question}"
)

def choose_answer_variant(answer: str) -> str:
    answer = str(answer).strip()
    if "\\boxed{" in answer:
        return answer
    return f"The final answer is \\boxed{{{answer}}}."

class BoxedReasoningDataset(Dataset):
    def __init__(self, df_in: pd.DataFrame, tokenizer):
        self.df = df_in.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        prompt = PROMPT_TEMPLATE.format(question=row["prompt"])
        answer = choose_answer_variant(row["answer"])

        prompt_text = (
            f"<|system|>\n{SYSTEM_PROMPT}\n"
            f"<|user|>\n{prompt}\n"
            f"<|assistant|>\n"
        )

        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_PROMPT_TOKENS,
        )["input_ids"]

        answer_ids = self.tokenizer(
            answer,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_ANSWER_TOKENS,
        )["input_ids"]

        eos_id = self.tokenizer.eos_token_id
        input_ids = prompt_ids + answer_ids + ([eos_id] if eos_id is not None else [])
        labels = [-100] * len(prompt_ids) + answer_ids + ([eos_id] if eos_id is not None else [])

        input_ids = input_ids[:MAX_TOTAL_TOKENS]
        labels = labels[:MAX_TOTAL_TOKENS]
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

class DynamicPadCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)

        def pad_tensor(t, pad_value):
            if len(t) == max_len:
                return t
            pad = torch.full((max_len - len(t),), pad_value, dtype=t.dtype)
            return torch.cat([t, pad], dim=0)

        return {
            "input_ids": torch.stack([pad_tensor(x["input_ids"], self.pad_token_id) for x in features]),
            "attention_mask": torch.stack([pad_tensor(x["attention_mask"], 0) for x in features]),
            "labels": torch.stack([pad_tensor(x["labels"], -100) for x in features]),
        }

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU is required.")

print("gpu:", torch.cuda.get_device_name(0))
print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_dataset = BoxedReasoningDataset(train_df, tokenizer)
val_dataset = BoxedReasoningDataset(val_df, tokenizer)
collator = DynamicPadCollator(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=SAVE_STEPS,
    eval_strategy="steps",
    save_strategy="steps",
    bf16=True,
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

trainer.train()

# Save adapter files, including adapter_config.json
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved files:", os.listdir(OUTPUT_DIR))
print("adapter_config.json exists:", os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json")))

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 100)

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')